# Do ECG and skin conductance respond together?

**Before you begin:** Request access to the [HABS dataset](https://huggingface.co/datasets/HABSteam/multimodal-eeg-emotion). Download day-1 preprocessed emotion-video BDFs and matching `.record_map` files for the participants you want to study, plus `labels_train.csv`. Place them under `sample/PREPROCESSED/P*/` and `sample/`, respectively, or change `DATA_ROOT` below. See the [notebook README](README.md) for environment setup. This notebook does not download data.

A small multimodal research example: **ECG + GSR features → CCA → held-out participant tests → interpretation**.

**Question:** Do ECG and skin conductance (GSR) share trial-specific information during emotional videos that generalizes to unseen participants?

1. Extract ECG and GSR features from 10 s video trials and inspect signal quality.
2. Learn CCA representations without arousal labels and inspect what they encode.
3. Test cross-modal correspondence on unseen participants against shuffled trials.
4. Use a simple arousal probe, interpret the results and suggest next steps.

**Trials and ratings:** BDF event markers define video intervals; `labels_train.csv` provides optional self-reported arousal for each trial. Ratings do not enter CCA training, and rating periods are excluded.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyedflib
from IPython.display import display
from scipy.signal import butter, find_peaks, sosfiltfilt
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut

# Same working-directory convention as the other starting notebooks.
cwd = Path.cwd().resolve()
if (cwd / "starting_notebooks" / "bdf_loader.py").is_file():
    ROOT = cwd
elif cwd.name == "starting_notebooks" and (cwd / "bdf_loader.py").is_file():
    ROOT = cwd.parent
else:
    raise RuntimeError("Start the kernel in the repository root or starting_notebooks/.")

if str(ROOT / "starting_notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "starting_notebooks"))
from bdf_loader import uV_to_uS

mne.set_log_level("ERROR")


## 1 · Select trials

Use any subset of locally available participants (including all of them), keeping **one 10 s video interval per observation**. BDF markers define the intervals; the trial-label table supplies optional self-reported arousal. We exclude rating periods and do not fill missing ratings.

**Reproducible example:** One exploratory run selected eight recordings from cohort 1: **P120, P124, P126, P128, P129, P131, P137, P138**. After quality screening, six were retained (P120, P126, P128, P129, P131, P137). Set `PARTICIPANT_IDS = REFERENCE_PARTICIPANTS` below to repeat that selection; `None` uses **all** locally available day-1 emotion-video recordings. Report *both* selected and retained participants, because results depend on the subset and its size. At least two usable participants are needed for leave-one-participant-out evaluation.

The preceding 2 s provide a GSR baseline, although they may contain activity from the previous rating. GSR responses are delayed, so this simple alignment is an assumption to revisit.

GSR is stored in **µV** in the BDF and converted to **µS** using the calibration function in the shared `bdf_loader.py`.


In [ ]:
DATA_ROOT = ROOT / "sample"                       # Or Path("/your/local/dataset")
BDF_ROOT = DATA_ROOT / "PREPROCESSED"
TRIAL_S = 10.0       # analysed window after video onset (s)
BASELINE_S = 2.0     # GSR reference window before video onset (s)
MIN_TRIALS = 10      # quality threshold (usable trials per participant; adjustable)
REFERENCE_PARTICIPANTS = ("P120", "P124", "P126", "P128", "P129", "P131", "P137", "P138")
PARTICIPANT_IDS = None  # None: every locally available day-1 participant; or set an ID list

# No participant-count cap: None selects every locally available day-1 recording.
# Explicit IDs reproduce a subset and fail clearly if any BDF is missing.
available_bdfs = sorted(BDF_ROOT.glob("P*/P*_day-1_task-emotion-video_multimodal.bdf"))
if PARTICIPANT_IDS is None:
    bdfs = available_bdfs
else:
    by_id = {bdf.parent.name: bdf for bdf in available_bdfs}
    missing = sorted(set(PARTICIPANT_IDS) - set(by_id))
    if missing:
        raise FileNotFoundError(f"Missing requested emotion-video BDFs in {BDF_ROOT}: {missing}")
    bdfs = [by_id[participant] for participant in PARTICIPANT_IDS]
if not bdfs:
    raise FileNotFoundError(f"No selected emotion-video BDFs found under {BDF_ROOT}.")

# The current release uses labels_train.csv; the older name also works.
labels_path = DATA_ROOT / "labels_train.csv"
if not labels_path.is_file():
    raise FileNotFoundError(f"Download labels_train.csv to {DATA_ROOT}.")

# Use the supplied marker lookup to pair video onsets and offsets.
marker_key = pd.read_csv(ROOT / "starting_notebooks" / "markers.csv")
video_key = marker_key.loc[marker_key["Start Marker"] >= 10100].set_index("Start Marker")
end_marker = video_key["End Marker"].to_dict()

labels = pd.read_csv(labels_path, na_values=["TESTSUBJECT"], usecols=[
    "cohort", "participant_id", "day", "start_marker", "rating_arousal"
])
labels["participant_id"] = labels["participant_id"].astype(str).str.strip()
labels["day"] = pd.to_numeric(labels["day"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
labels["start_marker"] = pd.to_numeric(labels["start_marker"], errors="coerce")
labels["rating_arousal"] = pd.to_numeric(labels["rating_arousal"], errors="coerce")
labels = labels.dropna(subset=["day", "start_marker"])
labels[["day", "start_marker"]] = labels[["day", "start_marker"]].astype(int)
keys = ["participant_id", "day", "start_marker"]
if labels.duplicated(keys).any():
    raise ValueError("Ambiguous trial markers in the label table; check participant/day/trial identifiers.")

# Report the selected IDs and their cohorts; local-file selection is not random.
selected = [bdf.parent.name for bdf in bdfs]
cohorts = labels.loc[labels.participant_id.isin(selected)].groupby("participant_id").cohort.first()
print(f"Selected {len(bdfs)} recordings: {', '.join(selected)}")
print("Participants per cohort:", cohorts.value_counts().sort_index().to_dict())


## 2 · Extract features

Here we extract some simple features. Feel free to consult the literature and try more complex methods.

For **ECG**, compute median heart rate, RR-interval variation and early-to-late heart-rate change (the 10 s estimate is not clinical HRV). For **GSR**, compute mean and maximum baseline-relative conductance change, plus conductance slope.

We read only the ECG and GSR channels to avoid loading the full EEG recording. Since direct pyEDFlib reads return GSR in **µV**, we call the shared `uV_to_uS` function once, before calculating features in **µS** and **µS/s**. (`load_bdf` applies this conversion automatically when used instead.)


In [ ]:
ECG_COLS = ["ecg_hr", "ecg_rr_sd", "ecg_hr_change"]
GSR_COLS = ["gsr_mean_delta", "gsr_max_delta", "gsr_slope"]


def detect_beats(ecg, fs):
    """Approximate R-peak positions (samples): 5–20 Hz band-pass, rectify, smooth, find peaks."""
    sos = butter(2, (5, 20), btype="bandpass", fs=fs, output="sos")
    # Rectification also handles an inverted ECG trace.
    width = max(1, round(0.08 * fs))
    envelope = np.convolve(np.abs(sosfiltfilt(sos, ecg)), np.ones(width) / width, mode="same")
    peaks, _ = find_peaks(envelope, distance=round(0.4 * fs), prominence=np.std(envelope))
    return peaks


def ecg_features(ecg, fs):
    """Heart rate, RR variability and heart-rate change in one trial, or None if unreliable."""
    peaks = detect_beats(ecg, fs)
    rr = np.diff(peaks) / fs
    plausible = (rr >= 0.4) & (rr <= 1.5)          # 40–150 beats per minute
    hr, beat_s = 60 / rr[plausible], peaks[1:][plausible] / fs
    late = beat_s >= TRIAL_S / 2
    if plausible.sum() < 6 or late.all() or not late.any():
        return None                                 # no trustworthy estimate from this window
    return {"ecg_hr": np.median(hr),
            "ecg_rr_sd": 1000 * np.std(rr[plausible]),
            "ecg_hr_change": hr[late].mean() - hr[~late].mean()}


def gsr_features(gsr, fs):
    """Baseline-relative conductance change (µS) and slope (µS/s)."""
    n_baseline = round(BASELINE_S * fs)
    response = gsr[n_baseline:] - gsr[:n_baseline].mean()
    seconds = np.arange(len(response)) / fs
    return {"gsr_mean_delta": response[seconds >= 2].mean(),  # exclude first 2 s of response
            "gsr_max_delta": response.max(),
            "gsr_slope": np.polyfit(seconds[::20], response[::20], 1)[0]}


def video_intervals(bdf):
    """Pair each video start marker with the immediately following end marker."""
    raw = mne.io.read_raw_bdf(bdf, preload=False, verbose="ERROR")
    events = pd.DataFrame({
        "time": raw.annotations.onset - raw.first_time,
        "code": pd.to_numeric(raw.annotations.description, errors="coerce"),
    }).dropna().reset_index(drop=True)
    raw.close()
    pairs = []
    for i in range(len(events) - 1):
        start, end = events.iloc[i], events.iloc[i + 1]
        marker = int(start.code)
        if marker in end_marker and end.code == end_marker[marker] and end.time > start.time:
            pairs.append((float(start.time), float(end.time), marker))
    return pairs

In [ ]:
rows, log, gsr_overview, gsr_summary, ecg_example = [], [], {}, [], None
for bdf in bdfs:
    participant = bdf.parent.name
    intervals = video_intervals(bdf)
    with pyedflib.EdfReader(str(bdf)) as reader:
        names = reader.getSignalLabels()
        if "ECG" not in names or "GSR" not in names:
            raise ValueError(f"Missing ECG/GSR in {bdf.name}; inspect its record_map.")
        ecg_id, gsr_id = names.index("ECG"), names.index("GSR")
        ecg_fs, gsr_fs = reader.getSampleFrequency(ecg_id), reader.getSampleFrequency(gsr_id)
        gsr_unit = reader.getPhysicalDimension(gsr_id).strip().replace("µ", "u").replace("μ", "u")
        if gsr_unit != "uV":
            raise ValueError(f"Expected GSR in uV, got {gsr_unit!r} in {bdf.name}.")

        # Direct channel reading is memory-efficient; conversion happens exactly once.
        gsr_uV = reader.readSignal(gsr_id)
        gsr_uS = uV_to_uS(gsr_uV)
        floor, ceiling = gsr_uV.min(), gsr_uV.max()
        gsr_overview[participant] = (gsr_uS[::100], 100 / gsr_fs, intervals)
        gsr_summary.append({"participant": participant, "BDF unit": gsr_unit,
                            "GSR min (µS)": gsr_uS.min(), "GSR median (µS)": np.median(gsr_uS),
                            "GSR max (µS)": gsr_uS.max()})

        for order, (start_s, end_s, marker) in enumerate(intervals):
            features = None
            if end_s - start_s < TRIAL_S - 0.1:
                status = "video shorter than 10 s"
            elif start_s < BASELINE_S:
                status = "no pre-video baseline"
            elif start_s + TRIAL_S > len(gsr_uS) / gsr_fs:
                status = "window exceeds recording"
            else:
                first = round((start_s - BASELINE_S) * gsr_fs)
                last = round((start_s + TRIAL_S) * gsr_fs)
                gsr = gsr_uS[first:last]
                raw_window = gsr_uV[first:last]
                if not np.isfinite(gsr).all():
                    status = "GSR non-finite"
                elif (np.mean(np.diff(gsr) == 0) > 0.5
                      or np.mean((raw_window <= floor) | (raw_window >= ceiling)) > 0.01):
                    status = "GSR flat or clipped"
                else:
                    ecg = reader.readSignal(ecg_id, round(start_s * ecg_fs), round(TRIAL_S * ecg_fs))
                    features = ecg_features(ecg, ecg_fs)
                    status = "kept" if features else "too few ECG beats"
            log.append({"participant_id": participant, "start_marker": marker, "status": status})
            if features:
                rows.append({"participant_id": participant, "day": 1, "start_marker": marker,
                             "trial_order": order, **features, **gsr_features(gsr, gsr_fs)})
                if ecg_example is None:
                    ecg_example = (participant, marker, ecg, ecg_fs)

log = pd.DataFrame(log)
if not rows:
    raise RuntimeError("No valid trials: inspect the signal-quality exclusions.")
trials = pd.DataFrame(rows).merge(labels.drop(columns="cohort"), on=keys, how="left",
                                  validate="one_to_one", indicator="in_labels")
if not np.isfinite(trials[ECG_COLS + GSR_COLS].to_numpy()).all():
    raise ValueError("Non-finite features: inspect the trial windows.")
print(f"Video intervals found: {len(log)}; usable trials: {len(trials)}")


## 3 · Check signal quality

Inspect trial exclusions, calibrated GSR levels and an example ECG beat detection. The shared loader's calibration converts GSR from **µV to µS**; the values below are skin conductance, not sensor voltage.


In [ ]:
print("Trials by participant and quality decision:")
display(pd.crosstab(log.participant_id, log.status))
print("GSR conductance after calibration:")
display(pd.DataFrame(gsr_summary).round(3))

# Count every selected participant, including those with zero usable trials.
usable = trials.groupby("participant_id").size()
dropped = {p: int(usable.get(p, 0)) for p in selected if usable.get(p, 0) < MIN_TRIALS}
if dropped:
    print("Dropped participants (usable trials):", dropped)
trials = trials.loc[~trials.participant_id.isin(dropped)].reset_index(drop=True)
retained = sorted(trials.participant_id.unique())
print(f"Kept {len(trials)} trials / {len(retained)} participants; "
      f"{trials.rating_arousal.notna().sum()} have arousal ratings.")
print("Retained participants:", ", ".join(retained))
print("Retained participants per cohort:", cohorts.reindex(retained).value_counts().sort_index().to_dict())
if len(retained) < 2:
    raise RuntimeError("Leave-one-participant-out CCA requires at least two usable participants.")

# Inspect a few GSR recordings and one example ECG trace.
bad = log.loc[log.status == "GSR flat or clipped", "participant_id"]
example_ids = list(dict.fromkeys(
    (bad.iloc[:1].tolist()) + [next(iter(gsr_overview)), list(gsr_overview)[-1]]
))
fig, axes = plt.subplots(len(example_ids) + 1, 1, figsize=(9, 2 * len(example_ids) + 2),
                         gridspec_kw={"height_ratios": [1] * len(example_ids) + [1.5]})
for ax, participant in zip(axes, example_ids):
    gsr, step_s, intervals = gsr_overview[participant]
    ax.plot(np.arange(len(gsr)) * step_s, gsr, lw=0.7)
    for start_s, end_s, _ in intervals:
        ax.axvspan(start_s, end_s, color="tab:orange", alpha=0.15)
    ax.set(ylabel=f"{participant} (µS)", xticks=[])
axes[0].set_title("Calibrated GSR (µS); shaded intervals indicate videos")
participant, marker, ecg, fs = ecg_example
peaks, seconds = detect_beats(ecg, fs), np.arange(len(ecg)) / fs
axes[-1].plot(seconds, ecg - np.median(ecg), lw=0.7)
axes[-1].plot(seconds[peaks], ecg[peaks] - np.median(ecg), "rx")
axes[-1].set(xlabel="Seconds after video onset", yticks=[],
             title=f"Example ECG beat detection: {participant}, marker {marker}")
plt.tight_layout()
plt.show()


## 4 · Fit shared representations

Standardize features **within each participant** to focus on trial-to-trial variation. This uses all *unlabelled* trials of a held-out participant as calibration: the CCA model transfers across participants, but this is **not** strictly calibration-free prediction.

For each leave-one-participant-out fold, fit CCA and PCA using the other participants. CCA learns correlated ECG/GSR projections; their scaled average defines a convenient joint embedding. PCA and original features are comparisons.

**Note:** Here, we chose CCA as one of the simplest methods for learning a latent representation. Feel free to explore more sophisticated approaches involving deep learning architectures, but always benchmark them against simpler baselines—greater complexity does not necessarily mean better results.

In [ ]:
z = trials.groupby("participant_id")[ECG_COLS + GSR_COLS].transform(lambda x: (x - x.mean()) / x.std())
x_ecg, x_gsr = z[ECG_COLS].to_numpy(), z[GSR_COLS].to_numpy()
x_both = np.column_stack([x_ecg, x_gsr])
if not np.isfinite(x_both).all():
    raise ValueError("A feature is constant within a participant; inspect the summary table.")

groups = trials.participant_id.to_numpy()
folds = list(LeaveOneGroupOut().split(x_both, groups=groups))
assert all(set(groups[train]).isdisjoint(groups[test]) for train, test in folds)


def fit_representations(train):
    """Fit PCA and CCA on the training participants only (no labels)."""
    pca = PCA(n_components=2).fit(x_both[train])
    cca = CCA(n_components=2, scale=False, max_iter=1000).fit(x_ecg[train], x_gsr[train])
    u_train, v_train = cca.transform(x_ecg[train], x_gsr[train])
    u_sd, v_sd = u_train.std(axis=0), v_train.std(axis=0)

    def transform(rows):
        """Every feature set compared later, for the given trial rows."""
        u, v = cca.transform(x_ecg[rows], x_gsr[rows])
        return {"ECG only": x_ecg[rows], "GSR only": x_gsr[rows], "Concatenation": x_both[rows],
                "PCA (2D)": pca.transform(x_both[rows]),
                "CCA shared (2D)": (u / u_sd + v / v_sd) / 2}
    return cca, transform


models = [fit_representations(train) for train, _ in folds]
print(f"{len(folds)} folds: each tests one participant on a model fitted to the other {len(folds) - 1}.")

## 5 · Interpret the learned representation

For **one fitted CCA model**, correlate the original features with their respective ECG or GSR score on the first canonical component. This illustrates what the representation captures *without using arousal labels*.

Feature correlations are descriptive, not causal contributions. The component sign is arbitrary, and this plot alone does not establish cross-modal correspondence.

In [ ]:
# Inspect one model on its held-out participant; do not mix axes across folds.
(_, test), (cca, _) = folds[0], models[0]
ecg_score, gsr_score = cca.transform(x_ecg[test], x_gsr[test])

feature_names = ["ECG · Heart rate", "ECG · RR variation", "ECG · HR change",
                 "GSR · Mean change", "GSR · Max change", "GSR · Slope"]
feature_r = (
    [np.corrcoef(x_ecg[test, i], ecg_score[:, 0])[0, 1] for i in range(len(ECG_COLS))]
    + [np.corrcoef(x_gsr[test, i], gsr_score[:, 0])[0, 1] for i in range(len(GSR_COLS))]
)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(feature_names, feature_r)
ax.axvline(0, color="gray", lw=0.8)
ax.set(xlim=(-1, 1), xlabel="Correlation with each modality's first CCA score",
       title=f"CCA component 1 · held-out {groups[test][0]}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6 · Test cross-modal correspondence

For every held-out participant, correlate the first ECG and GSR CCA projections on **matched trials**. Shuffle GSR trial order within that participant to build a control distribution.

Shuffling removes exact pairs, but does **not** rule out common session drift or carry-over effects. Compare participants individually and check whether one person dominates the group result.

In [ ]:
def correlation(x, y):
    return float(np.corrcoef(x, y)[0, 1])


N_SHUFFLES = 1000
rng = np.random.default_rng(42)
per_participant, null = [], []
for (train, test), (cca, _) in zip(folds, models):
    u, v = cca.transform(x_ecg[test], x_gsr[test])
    u, v = u[:, 0], v[:, 0]
    observed = correlation(u, v)
    shuffled = np.array([correlation(u, rng.permutation(v)) for _ in range(N_SHUFFLES)])
    null.append(shuffled)
    per_participant.append({"participant": groups[test][0], "trials": len(test), "matched r": observed,
                            "shuffled 2.5%": np.percentile(shuffled, 2.5),
                            "shuffled 97.5%": np.percentile(shuffled, 97.5)})
per_participant = pd.DataFrame(per_participant)

group_r, group_null = per_participant["matched r"].mean(), np.mean(null, axis=0)
group_p = (1 + np.sum(group_null >= group_r)) / (1 + N_SHUFFLES)
print(f"Mean matched r = {group_r:+.3f}; shuffled-pair p = {group_p:.3f} "
      "(exploratory; assumes exchangeable trials)")
print(f"Participants with positive r: {(per_participant['matched r'] > 0).sum()} / {len(per_participant)}")
print(f"Mean r without the strongest participant: "
      f"{per_participant['matched r'].drop(per_participant['matched r'].idxmax()).mean():+.3f}")

fig, ax = plt.subplots(figsize=(8, 3.4))
positions = np.arange(len(per_participant))
ax.vlines(positions, per_participant["shuffled 2.5%"], per_participant["shuffled 97.5%"],
          color="tab:blue", alpha=0.4, lw=6, label="Shuffled-pair range")
ax.scatter(positions, per_participant["matched r"], color="black", zorder=3, label="Matched trials")
ax.axhline(0, lw=0.8, color="gray")
ax.set_xticks(positions, per_participant.participant, rotation=45)
ax.set(ylabel="Matched ECG–GSR correlation", title="Held-out participants (CCA component 1)")
ax.legend(fontsize="small")
plt.tight_layout()
plt.show()

## 7 · Probe downstream utility

As an **exploratory downstream probe**, classify each trial as above/below its participant's median *reported* arousal. Compare ECG, GSR, concatenation, PCA and CCA with a fixed logistic regression and majority-class baseline.

The held-out participant's ratings are needed to define this relative target. It is **not** a deployable, calibration-free arousal predictor; the probe only checks whether the representation contains useful information.

In [ ]:
median_rating = trials.groupby("participant_id").rating_arousal.transform("median")
y = trials.rating_arousal.gt(median_rating).astype(float).where(trials.rating_arousal.notna())
labelled, target = y.notna().to_numpy(), y.to_numpy()

scores = {}
for (train, test), (_, transform) in zip(folds, models):
    train, test = train[labelled[train]], test[labelled[test]]
    if len(test) < 5 or len(set(target[test])) < 2:
        continue
    features_train, features_test = transform(train), transform(test)
    baseline = DummyClassifier(strategy="most_frequent").fit(x_both[train], target[train])
    scores.setdefault("Majority class", []).append(
        balanced_accuracy_score(target[test], baseline.predict(x_both[test])))
    for name in features_train:
        probe = LogisticRegression(class_weight="balanced", max_iter=1000)
        probe.fit(features_train[name], target[train])
        scores.setdefault(name, []).append(
            balanced_accuracy_score(target[test], probe.predict(features_test[name])))

print(f"{labelled.sum()} rated trials; {len(scores['Majority class'])} held-out participants scored")
display(pd.Series({name: f"{np.mean(values):.3f} ± {np.std(values):.3f}"
                   for name, values in scores.items()}).reindex([
    "Majority class", "ECG only", "GSR only", "Concatenation", "PCA (2D)", "CCA shared (2D)"
]).to_frame("Arousal · balanced accuracy"))

Think about what other approaches could be relevant for exploring the latent space?

## Interpretation and next steps

**Findings.** Which features are associated with the learned components? Is ECG/GSR correspondence stronger for matched than shuffled trials? Does the shared representation retain information related to reported arousal?

A shared representation does not need to outperform direct supervised prediction: its purpose is to capture structure learned **without arousal labels**. Positive and negative findings can both motivate follow-up experiments.

**Reproducibility.** Always report your experiment configuration (participants, exclusions, trials, ...) and validate generalizability of your solution. The example selection above contained eight BDFs but only **six evaluable participants** (142 rated trials); results from this small, single-cohort subset are exploratory, not established generalization.

**Research extensions.** Test whether the conclusions **generalize across larger and more diverse participant sets and cohorts**. Try different subset sizes and compositions, ideally with repeated random selections: do the correlations and arousal-probe scores remain stable? Also explore **different numbers of recording sessions per participant**. This notebook currently uses day 1 only; adding more sessions requires extending trial selection and keeping **all sessions of a participant in the same train/test group** to avoid leakage. 

There are many possible directions to extend the analysis:

* **Richer physiological feature sets.** Go beyond the few handcrafted ECG and GSR features used in this baseline. Explore additional temporal, statistical, spectral, and nonlinear features that might capture complementary physiological information.

* **More advanced feature extraction.** Investigate established and state-of-the-art methods for extracting informative physiological features, including approaches that better capture signal dynamics and temporal dependencies.

* **Additional modalities.** Extend the analysis beyond ECG and GSR by incorporating signals such as EEG, pupillometry, and respiration. Investigate whether additional modalities provide complementary information and improve the quality or generalizability of the learned representations.

* **More sophisticated representation learning.** CCA is only one of the simplest approaches to learning a shared latent space. Explore alternative methods, including deep learning architectures, self-supervised learning, multimodal contrastive learning, and pretrained foundation models.

* **Deeper latent-space analysis.** Go beyond cross-modal correlations and arousal prediction. Investigate the structure, interpretability, stability, and information content of the learned representations. For example, use linear probing with additional targets, such as valence, emotion categories, or other available annotations. Consider whether the representations capture meaningful physiological information or potentially confounding factors such as participant identity or recording conditions.

These are suggestions rather than a predefined roadmap. Choose a research question, formulate a hypothesis, and design an evaluation that can meaningfully test it. A well-controlled experiment with a simple method can be more valuable than a complex model without convincing validation.

**Resources:** [`DATASET.md`](../DATASET.md) · [scikit-learn CCA](https://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.CCA.html).
